In [5]:
import sys
sys.path.insert(0, '../')

from src.process import NeuroFeatureExtractor
from src.load_data import PatientLoader

import numpy as np

extractor = NeuroFeatureExtractor() 


file_path = "data/sub-1004"
loader = PatientLoader()
recording = loader.load(file_path)

[fetch_atlas_harvard_oxford] Dataset found in C:\Users\ainho\nilearn_data\fsl


D:\Universidad\6 infierno de Dante\neuroimagen\projecto_git\project2\Neuroimage-FinalProjectCode\src\process.py:15: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  self.resampled_atlas = image.resample_img(
D:\Universidad\6 infierno de Dante\neuroimagen\projecto_git\project2\Neuroimage-FinalProjectCode\src\process.py:15: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  self.resampled_atlas = image.resample_img(


In [22]:
import os
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from nilearn import masking, image
from nilearn.maskers import NiftiLabelsMasker

#We mpve to the data/ folder
os.chdir("data")

import os
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from nilearn import masking, image

# --- CONFIGURACIÓN PREVIA ---
X_grey_matter = []
y_true = []  # 0 para Jóvenes (sub-1xxx), 1 para Adultos (sub-2xxx)


for patient_dir in os.listdir("."):
    if not patient_dir.startswith("sub-"):
        continue 
        
    patient_id = patient_dir.split('-')[1]
    print(f"Processing subject: {patient_dir}...")
    
    try:
        # 1. Upload patient code)
        recording = loader.load(patient_dir)
        anat_img = recording.anat.img
        data = anat_img.get_fdata()
        
        # 2. Skull Stripping automatic (Brain Mask)
        brain_mask = masking.compute_brain_mask(anat_img)
        brain_indices = brain_mask.get_fdata() > 0
        voxels_inside_brain = data[brain_indices]
        
        # 3. Segmentation with GMM to isolate Grey Matter
        gmm = GaussianMixture(n_components=3, random_state=42)
        labels = gmm.fit_predict(voxels_inside_brain.reshape(-1, 1))
        
        # Order by intensity. Grey matter will be in the middle
        means = gmm.means_.flatten()
        sorted_tissue_indices = np.argsort(means)
        gm_label = sorted_tissue_indices[1]  # Materia Gris
        
        # Reconstruir la máscara 3D de Materia Gris
        gm_mask_data = np.zeros(data.shape)
        gm_mask_data[brain_indices] = (labels == gm_label).astype(int)
        
        # 4. Adapt atlas to our patient
        resampled_atlas = image.resample_to_img(
            source_img=extractor.atlas.maps,
            target_img=anat_img,
            interpolation='nearest'
        )
        atlas_data = resampled_atlas.get_fdata()
        
        # To have the same dimension for all the subjects
        num_labels = len(extractor.atlas.labels) - 1 
        
        gm_density_per_roi = []
        for label_idx in range(1, num_labels + 1):
            # Creamos una máscara para la región actual del atlas
            roi_mask = (atlas_data == label_idx)
            total_roi_voxels = np.sum(roi_mask)
            
            if total_roi_voxels > 0:
                # Proporción de materia gris real dentro de los vóxeles de esta ROI
                roi_gm_voxels = np.sum(gm_mask_data[roi_mask])
                density = roi_gm_voxels / total_roi_voxels
            else:
                # Si la región quedó fuera del FOV del paciente o está vacía, forzamos 0.0
                # Esto garantiza que el vector mantenga siempre la misma longitud
                density = 0.0
                
            gm_density_per_roi.append(density)
            
        # Guardamos el vector (que ahora sí medirá exactamente lo mismo para todos)
        X_grey_matter.append(np.array(gm_density_per_roi))
        
        # Guardamos la etiqueta real (Ground Truth)
        if patient_id.startswith('1'):
            y_true.append(0)  # Joven
        elif patient_id.startswith('2'):
            y_true.append(1)  # Adulto
            
    except Exception as e:
        print(f"Error processing subject {patient_dir}: {e}")

# Al ser todos los vectores idénticos en tamaño, esto funcionará a la perfección
X = np.array(X_grey_matter)
y_true = np.array(y_true)

print(f"\n¡Extraction finished. Matrix with shape: {X.shape}")

os.chdir("..")

# =========================================================================
# ENTRENAMIENTO DEL MODELO K-MEANS
# =========================================================================
print("\nTraining K-Means with 2 clusters...")

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)

# Alinear etiquetas automáticas de K-Means con vuestras clases reales (0=Joven, 1=Adulto)
acc_original = accuracy_score(y_true, y_pred)
acc_invertido = accuracy_score(y_true, 1 - y_pred)

if acc_invertido > acc_original:
    y_pred = 1 - y_pred
    print("Nota: Se han invertido las etiquetas de K-Means para alinearlas con la realidad.")

# =========================================================================
# EVALUACIÓN DE RESULTADOS
# =========================================================================
final_accuracy = accuracy_score(y_true, y_pred)
conf_matrix = confusion_matrix(y_true, y_pred)

print("\n================ RESULTADOS DEL K-MEANS ================")
print(f"Precisión (Accuracy) del agrupamiento: {final_accuracy * 100:.2f}%")
print("\nMatriz de Confusión:")
print("                 Predicho Joven   Predicho Adulto")
print(f"Real Joven (1xx):      {conf_matrix[0][0]}                {conf_matrix[0][1]}")
print(f"Real Adulto (2xx):     {conf_matrix[1][0]}                {conf_matrix[1][1]}")
print("========================================================")

Processing subject: sub-1004...


C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled_atlas = image.resample_to_img(
C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled_atlas = image.resample_to_img(


Processing subject: sub-1005...


C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled_atlas = image.resample_to_img(
C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled_atlas = image.resample_to_img(


Processing subject: sub-1006...


...

C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled_atlas = image.resample_to_img(
C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled_atlas = image.resample_to_img(


Processing subject: sub-2038...


C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  resampled_atlas = image.resample_to_img(
C:\Users\ainho\AppData\Local\Temp\ipykernel_19856\1719551111.py:56: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  resampled_atlas = image.resample_to_img(



¡Extraction finished. Matrix with shape: (62, 96)

Training K-Means with 2 clusters...

================ RESULTADOS DEL K-MEANS ================
Precisión (Accuracy) del agrupamiento: 58.06%

Matriz de Confusión:
                 Predicho Joven   Predicho Adulto
Real Joven (1xx):      16                18
Real Adulto (2xx):     8                20


D:\Programas\anaconda\envs\neuroimage\lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
